# Deep Research End-to-End Workflow - May 2026 Release

## Overview
This notebook demonstrates the complete end-to-end workflow for deep research analysis, integrating multiple AI agents to provide comprehensive insights and recommendations.

## Solution Architecture

```mermaid
graph TD
    A[Raja - Anomaly<br/>COC_CMN_TRND_MTRC_INSGHT_STG<br/><br/>Top 3 States | Top 5 Providers | Top 5 DRG<br/>Top 5 Provider/DRG within each state]

    B[Rajib - Deep Dive<br/>COC_CMN_DATA_INSGHT_STG<br/>Deepdive & KeyInsight Summary<br/><br/>For each state: DRG, auth approvals, metrics]

    C[Semantic Model<br/>Claim, Member, Auth, Network, PI,<br/>Metric Definition]

    D[Correlation Agent<br/>Additional dimensions<br/>Product, Facility, Diagnosis, Procedure, Modifier<br/><br/>3 States | 5 DRG | 5 Providers]

    E[Pattern Analysis Agent<br/>Aggregates and finds common patterns<br/><br/>Pattern-1<br/>Pattern-N]

    F[Reimbursement Policy Agent - SME<br/><br/>Reimbursement Policy<br/>Reimbursement Policy]

    G[Recommendation Agent<br/>Aggregates all patterns and explanations<br/>to generate recommendations]

    H[Final Recommendations]

    A --> B
    B --> C
    B --> D
    D --> E
    E --> F
    E --> G
    F --> G
    G --> H

    style A fill:#e1f5ff,stroke:#00b8b8,stroke-width:2px,color:#173b7a
    style B fill:#e1f5ff,stroke:#00b8b8,stroke-width:2px,color:#173b7a
    style C fill:#e8eef9,stroke:#173b7a,stroke-width:2px,color:#173b7a
    style D fill:#ffffff,stroke:#00b8b8,stroke-width:2px,color:#173b7a
    style E fill:#fff4e1,stroke:#00b8b8,stroke-width:2px,color:#173b7a
    style F fill:#fff4e1,stroke:#00b8b8,stroke-width:2px,color:#173b7a
    style G fill:#e8f5e9,stroke:#00b8b8,stroke-width:2px,color:#173b7a
    style H fill:#e8f5e9,stroke:#173b7a,stroke-width:2px,color:#173b7a
```

## Workflow Steps

1. **Data Loading**: Load insights from Rajib's anomaly detection (KEY_INSIGHT and DEEP_DIVE)
2. **Pattern Analysis**: Extract actionable patterns from DEEP_DIVE reports
3. **Correlation Analysis**: Find additional dimensions (procedures, modifiers) driving each pattern
4. **Policy Extraction**: Retrieve relevant reimbursement policies for identified codes
5. **Recommendation Synthesis**: Aggregate findings to generate actionable recommendations

## API Endpoints
- Pattern Agent: `POST http://localhost:8000/agents/pattern_agent`
- Correlation Agent: `POST http://localhost:8000/agents/correlation`
- Reimbursement Policy Agent: `POST http://localhost:8000/agents/reimbursement_policy`
- Recommendation Agent: `POST http://localhost:8000/agents/recommendation_synthesis`

# Setup & Initialization

In [1]:
%load_ext autoreload
%autoreload 2

In [7]:
import sys
import os
import copy
import pandas as pd
import json
import requests
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown, HTML

# Configure pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

print(f"Python: {sys.executable}")
print(f"Version: {sys.version}")

Python: c:\projects\coc\dev\idiscovery-deep-research\.venv\Scripts\python.exe
Version: 3.13.13 (main, May 10 2026, 19:31:06) [MSC v.1944 64 bit (AMD64)]


In [8]:
# Load environment variables
project_root = Path.cwd().parent.parent.parent
env_path = project_root / ".env"
load_dotenv(env_path)

print(f"Project root: {project_root}")
print(f"Environment loaded: {env_path.exists()}")

Project root: c:\projects\coc\dev\idiscovery-deep-research
Environment loaded: True


In [9]:
from deep_research_utils import SnowparkHelper, EHAPBase
from deep_research_utils.app_constant import AppConstants
from langchain_openai import ChatOpenAI

2026-05-29 20:07:09,591 - policy_extractor.system - INFO - === Policy Extractor Logging Initialized ===
2026-05-29 20:07:09,592 - policy_extractor.system - INFO - Log directory: c:\projects\coc\dev\idiscovery-deep-research\notebooks\examples\2026-05\logs
2026-05-29 20:07:09,592 - policy_extractor.system - INFO - Max file size: 50.0MB
2026-05-29 20:07:09,593 - policy_extractor.system - INFO - Backup count: 10
2026-05-29 20:07:09,594 - policy_extractor.system - INFO - Console output enabled: True
2026-05-29 20:07:09,596 - policy_extractor.system - INFO - Console log level: INFO
2026-05-29 20:07:09,598 - policy_extractor.system - INFO - Console stream: stdout
2026-05-29 20:07:09,600 - policy_extractor.system - INFO - Process ID: 22788
2026-05-29 20:07:09,602 - policy_extractor.system - INFO - Component log levels:
2026-05-29 20:07:09,603 - policy_extractor.system - INFO -   policy_extractor.snowflake_store: WARNING
2026-05-29 20:07:09,604 - policy_extractor.system - INFO -   policy_extrac

In [10]:
# Initialize Snowpark connection
snowpark_programmatic_connection_parameters = {
    "account": os.environ["SNOWFLAKE_ACCOUNT"],
    "user": os.environ["SNOWFLAKE_USER"],
    "password": os.environ["SNOWFLAKE_SECRET"],
    "warehouse": os.environ["SNOWFLAKE_WAREHOUSE"],
    "database": os.environ["SNOWFLAKE_DATABASE"],
    "schema": os.environ["SNOWFLAKE_SCHEMA"]
}

snowpark = SnowparkHelper(
    connection_type="programmatic",
    batch_size=10000,
    max_workers=6,
    enable_metrics=True,
    connection_pool_size=4,
    **snowpark_programmatic_connection_parameters
)

print("✓ Snowpark connection established")

2026-05-29 20:07:11,002 - deep_research_utils.snowflake_helper - INFO - 🔑 Configuring PROGRAMMATIC connection (password-based authentication)
2026-05-29 20:07:36,508 - deep_research_utils.snowflake_helper - ERROR - Error creating Snowflake session: 250001 (08004): Failed to get authentication by OKTA: 401: Unauthorized


DatabaseError: 250001 (08004): Failed to get authentication by OKTA: 401: Unauthorized

In [ ]:
# Initialize EHAP authentication
EHAP = EHAPBase(
    base_url=os.environ.get("EHAP_BASE_URL"),
    client_id=os.environ.get("EHAP_CLIENT_ID"),
    client_secret=os.environ.get("EHAP_CLIENT_SECRET"),
    verify=os.environ.get("SSL_CERT_FILE")
)

print("✓ EHAP authentication initialized")

✓ EHAP authentication initialized


In [8]:
# Initialize LLM
from langchain_core.runnables import ConfigurableField
llm = ChatOpenAI(
    model=AppConstants.EHAP_LLM_MODEL,
    api_key=EHAP.get_token(),
    extra_body={
        "reasoning_effort": "medium",
        "summary": None
    }
).configurable_fields(
    openai_api_key=ConfigurableField(id="user_api_key")
)

llm_low_reasoning = ChatOpenAI(
    model=AppConstants.EHAP_LLM_MODEL,
    api_key=EHAP.get_token(),
    extra_body={
        "reasoning_effort": "low",
        "summary": None
    }
).configurable_fields(
    openai_api_key=ConfigurableField(id="user_api_key")
)

print("✓ LLM initialized")

2026-05-28 15:06:01,715 - deep_research_utils.ehap - INFO - Requesting new access token from https://api.horizon.elevancehealth.com/v2/oauth2/token with client_id: VOlF2INQArWlEG4icdjyI7js5ICMuwfM
2026-05-28 15:06:01,846 - deep_research_utils.ehap - INFO - Access token generated successfully.
**TOKEN** **TOKEN** **TOKEN** 


/Users/AH45807/project/idiscovery-deep-research/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.horizon.elevancehealth.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


✓ LLM initialized


In [11]:
# API Configuration
API_BASE_URL = "http://localhost:8000"
CONVERSATION_ID = f"tutorial_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

print(f"API Base URL: {API_BASE_URL}")
print(f"Conversation ID: {CONVERSATION_ID}")

API Base URL: http://localhost:8000
Conversation ID: tutorial_20260529_200741


# Step 1: Load Insights Data

Load the insights generated by Rajib's anomaly detection system. This includes both KEY_INSIGHT and DEEP_DIVE summaries.


This table is stored in Snowflake:
```sql
select * from U01_COC.COC_DTI_STG.coc_cmn_data_insght_stg
```

In [11]:
# Load insights data
# Note: Update this path to match your local environment
insights_csv_path = project_root / "notebooks/examples/2026-05/2026-05-18-UAT-data-insight-stg.csv"

# Alternative: You can also query directly from Snowflake if available
# df_insights = snowpark.session.table("COC_CMN_DATA_INSGHT_STG").to_pandas()

df_insights = pd.read_csv(insights_csv_path)
focus_hcc = "IP AUTH" # for 2026-05 release this is the HCC of interest
df_insights = df_insights[df_insights.STATSCL_MDL_CD == focus_hcc]
print(f"Loaded insights shape: {df_insights.shape}")
print(f"\nInsight types:")
print(df_insights.INSGHT_TYPE_NM.value_counts())
print(f"\nColumns: {list(df_insights.columns)}")

Loaded insights shape: (24, 18)

Insight types:
INSGHT_TYPE_NM
DEEP_DIVE      12
KEY_INSIGHT    12
Name: count, dtype: int64

Columns: ['EDL_LOAD_DTM', 'EDL_RUN_ID', 'EDL_SOR_CD', 'KF_TMS', 'EDL_SCRTY_LVL_CD', 'EDL_LOB_CD', 'EDL_EXTRNL_LOAD_CD', 'EDL_CREAT_DTM', 'EDL_INCRMNTL_LOAD_DTM', 'SNAP_YEAR_MNTH_NBR', 'TRND_TM_PRD_END_MNTH_NBR', 'TRND_TM_PRD_CD', 'LOB_CD', 'LOB_SHRT_DESC', 'STATSCL_MDL_CD', 'INSGHT_TYPE_NM', 'JSON_TXT', 'OFSHR_EXCLSN_SOR_CD']


# Step 2: Process First Row - End-to-End Example

We'll walk through the complete workflow for the first row to demonstrate each agent interaction.

In [12]:
for snap_year_mnth_nbr in df_insights.SNAP_YEAR_MNTH_NBR.unique():
    for trnd_tm_prd_end_mnth_nbr in df_insights.TRND_TM_PRD_END_MNTH_NBR.unique():
        for trnd_tm_prd_cd in df_insights.TRND_TM_PRD_CD.unique():
            for lob_shrt_desc in df_insights.LOB_SHRT_DESC.unique():
                for statscl_mdl_cd in df_insights.STATSCL_MDL_CD.unique():
                    print(f"{snap_year_mnth_nbr}, {trnd_tm_prd_end_mnth_nbr}, {trnd_tm_prd_cd}, {lob_shrt_desc}, {statscl_mdl_cd}")
                    break
                break
            break
        break
    break

202604, 202601, R3, Commercial, IP AUTH


In [13]:
first_anomaly = df_insights[(df_insights.SNAP_YEAR_MNTH_NBR == snap_year_mnth_nbr) &
                              (df_insights.TRND_TM_PRD_END_MNTH_NBR == trnd_tm_prd_end_mnth_nbr) &
                              (df_insights.TRND_TM_PRD_CD == trnd_tm_prd_cd) &
                              (df_insights.LOB_SHRT_DESC == lob_shrt_desc) & 
                              (df_insights.STATSCL_MDL_CD == statscl_mdl_cd) &
                              (df_insights.INSGHT_TYPE_NM == 'KEY_INSIGHT')]
anomaly_json = json.loads(json.loads(first_anomaly.JSON_TXT.iloc[0]))
anomaly_json

{'whats_happening': '',
 'top_contributors': {'provider_trends': [{'name': 'EMORY HILLANDALE HOSPITAL',
    'insight': 'Experienced a substantial increase in authorizations',
    'percentage_change': '+490%'},
   {'name': 'EMORY DECATUR HOSPITAL',
    'insight': 'Showed a significant rise in authorization counts',
    'percentage_change': '+239%'},
   {'name': 'YALE NEW HAVEN HOSPITAL',
    'insight': 'Noted a considerable increase in authorizations',
    'percentage_change': '+34%'},
   {'name': "HENRICO DOCTORS' HOSPITAL",
    'insight': 'Saw a moderate increase in authorization counts',
    'percentage_change': '+22%'}],
  'states': [{'name': 'ME',
    'insight': 'Had a significant increase in authorization counts',
    'percentage_change': '+52%'},
   {'name': 'CO',
    'insight': 'Experienced a notable rise in authorization counts',
    'percentage_change': '+14%'}],
  'drgs': [{'name': 'Ungroupable',
    'insight': 'Showed a substantial increase in authorizations',
    'percentag

In [14]:
first_deep_dive = df_insights[(df_insights.SNAP_YEAR_MNTH_NBR == snap_year_mnth_nbr) &
                              (df_insights.TRND_TM_PRD_END_MNTH_NBR == trnd_tm_prd_end_mnth_nbr) &
                              (df_insights.TRND_TM_PRD_CD == trnd_tm_prd_cd) &
                              (df_insights.LOB_SHRT_DESC == lob_shrt_desc) & 
                              (df_insights.STATSCL_MDL_CD == statscl_mdl_cd) &
                              (df_insights.INSGHT_TYPE_NM == 'DEEP_DIVE')]

deep_dive_json = json.loads(json.loads(first_deep_dive.JSON_TXT.iloc[0]))
deep_dive_json

{'report_title': 'IP Authorization Insights - R3 (Snap Month: 202604, Period End: 202601) - Commercial',
 'national_summary': {'description': 'The national total for Commercial IP authorizations R3 is 33,810 (AMFINR). This report covers the top variance drivers by State, DRG, and Provider for the AUTH_CNT metric.'},
 'top_state_drivers': {'section_title': 'TOP STATE DRIVERS',
  'states': [{'state_name': 'CT',
    'overview': 'CT has 1,802 auths out of 33,810 nationally (5.33% of the national total).',
    'medical_necessity_review_mix': 'In CT, of 1,802 authorizations, 95.45% require Medical Necessity review.',
    'service_driver': 'In CT, of total 1,802 authorizations, IP Med/Surg contributes 1,124 authorizations, accounting for 62.38%; IP BH contributes 511 authorizations, accounting for 28.36%; IP OB Dlvry NB contributes 115 authorizations, accounting for 6.38%; NF contributes 55 authorizations, accounting for 3.05%.',
    'authorization_status_mix': 'In CT, of the 1,802 authorizat

## Step 2.1: Correlation Agent/ Waterfall Agent

The Correlation Agent identifies additional dimensions (procedures, modifiers, etc.) that drive the observed patterns. It drill down to find the most significant factors contributing to the pattern.

Start API server:
```bash
source .venv/bin/activate && uvicorn packages.agents.src.agent_api:app --
reload --host 0.0.0.0 --port 8000
```

In [15]:
from datetime import datetime

def get_ecap_start_month(trnd_tm_prd_cd: str, trnd_tm_prd_end_mnth_nbr: int) -> int:
    """
    Compute start month (YYYYMM) for a given ECAP time period.

    Args:
        trnd_tm_prd_cd (int): One of ["R3", "R6", "R12", "YTD"]
        trnd_tm_prd_end_mnth_nbr (int): End month in YYYYMM format

    Returns:
        int: Start month in YYYYMM format
    """

    end_date = datetime.strptime(str(trnd_tm_prd_end_mnth_nbr), "%Y%m")

    def subtract_months(dt, months):
        year = dt.year
        month = dt.month - months

        while month <= 0:
            month += 12
            year -= 1

        return datetime(year, month, 1)

    if trnd_tm_prd_cd.startswith("R"):
        months = int(trnd_tm_prd_cd[1:])
        # standard rolling window (inclusive)
        start_date = subtract_months(end_date, months - 1)

    elif trnd_tm_prd_cd == "YTD":
        start_date = datetime(end_date.year, 1, 1)

    else:
        raise ValueError(f"Unsupported trnd_tm_prd_cd: {trnd_tm_prd_cd}")

    return int(start_date.strftime("%Y%m"))

# Test cases
print(get_ecap_start_month("R3", 202501))   # Expected: 202411
print(get_ecap_start_month("R6", 202501))   # Expected: 202408
print(get_ecap_start_month("R12", 202501))  # Expected: 202402
print(get_ecap_start_month("YTD", 202512))  # Expected: 202501

202411
202408
202402
202501


In [16]:
def convert_current_ecap_time_to_previous_year(current_period_start: int,
                                               current_period_end: int) -> tuple[int, int]:
    """
    Convert ECAP period (YYYYMM) to previous year period.

    Args:
        current_period_start (int): Start period in YYYYMM format
        current_period_end (int): End period in YYYYMM format

    Returns:
        tuple[int, int]: (previous_period_start, previous_period_end)
    """

    def shift_to_previous_year(period: int) -> int:
        year = period // 100
        month = period % 100

        if not (1 <= month <= 12):
            raise ValueError(f"Invalid month in period: {period}")

        return (year - 1) * 100 + month

    previous_period_start = shift_to_previous_year(current_period_start)
    previous_period_end = shift_to_previous_year(current_period_end)

    return previous_period_start, previous_period_end

In [17]:
# if you dont set them here, it will pick up the defaults from the configs/correlation_pattern/coc_ecap_ip_auth_sematic_view_with_samples.yaml
current_time_period_end = int(trnd_tm_prd_end_mnth_nbr) # critical step to convert the numpy to int64, else API will throw error
current_time_period_start = get_ecap_start_month(trnd_tm_prd_cd, current_time_period_end)
# print("Current period:", current_time_period_start, current_time_period_end)

previous_period_start, previous_period_end = convert_current_ecap_time_to_previous_year(current_time_period_start, current_time_period_end)
# print(f"Previous period: {previous_period_start} to {previous_period_end}")
conversation_id = f"tutorial-{statscl_mdl_cd}-{lob_shrt_desc}-{snap_year_mnth_nbr}-{trnd_tm_prd_cd}-{trnd_tm_prd_end_mnth_nbr}".replace(" ", "_")

correlation_agent_common_payload = {
    "conversation_id": conversation_id,
    "context":{
      "analysis_mode_parameters": {
        "drill_metric": ["expense_detail.total_paid"],
        "period": {
          "rolling_time_dimension": "expense_detail.incurred_month",
          "current_period": {
            "start_time": current_time_period_start,
            "end_time": current_time_period_end
          },
          "previous_period": {
            "start_time": previous_period_start,
            "end_time": previous_period_end
          }
        }
      },
      "filters": [
        {
          "field": "snap_month",
          "operator": "=",
          "value": int(snap_year_mnth_nbr),
          "source": "dimension_match"
        },
        {
          "field": "lob_description",
          "operator": "=",
          "value": lob_shrt_desc,
          "source": "dimension_match"
        }
      ]
    },

  }
# add the HCC filter
if statscl_mdl_cd == "IP AUTH":
  correlation_agent_common_payload["context"]["filters"].append({
        "field": "hcc_high",
        "operator": "=",
        "value": "IP",
        "source": "dimension_match"
      })
else:
    correlation_agent_common_payload["context"]["filters"].append({
        "field": "hcc_medium",
        "operator": "=",
        "value": statscl_mdl_cd,
        "source": "dimension_match"
    })
# correlation_agent_common_payload

In [ ]:
%%time
# Call Correlation Agent for all the states 
correlation_agent_url = f"{API_BASE_URL}/agents/correlation"
correlation_results = {
    "states": {},
    "providers": {},
    "drgs": {}
}
for state in anomaly_json["top_contributors"]["states"]:
    correlation_agent_payload = copy.deepcopy(correlation_agent_common_payload)
    correlation_agent_payload["query"] = f"Where did change happen for state {state['name']}? It {state['insight'].lower()} by {state['percentage_change']}"
    correlation_agent_payload["context"]["filters"].append({
        "field": "service_area_state",
        "operator": "=",
        "value": state["name"],
        "source": "dimension_match"
    })
    print(f"Calling Correlation Agent for state '{state['name']}'...")
    correlation_response = requests.post(correlation_agent_url, json=correlation_agent_payload)
    correlation_result = correlation_response.json()

    print(f"Status: {correlation_response.status_code}")
    print(f"Success: {correlation_result.get('status', False)}")
    correlation_results["states"][state["name"]] = correlation_result

# run for all provider_trends
for provider in anomaly_json["top_contributors"]["provider_trends"]:
    correlation_agent_payload = copy.deepcopy(correlation_agent_common_payload)
    correlation_agent_payload["query"] = f"Where did change happen for provider {provider['name']}? It {provider['insight'].lower()} by {provider['percentage_change']}"
    correlation_agent_payload["context"]["filters"].append({
        "field": "rendering_provider_name",
        "operator": "=",
        "value": provider["name"],
        "source": "dimension_match"
    })
    print(f"Calling Correlation Agent for provider '{provider['name']}'...")
    correlation_response = requests.post(correlation_agent_url, json=correlation_agent_payload)
    correlation_result = correlation_response.json()

    print(f"Status: {correlation_response.status_code}")
    print(f"Success: {correlation_result.get('status', False)}")
    correlation_results["providers"][provider["name"]] = correlation_result


#  run for all drgs
for drg in anomaly_json["top_contributors"]["drgs"]:
    correlation_agent_payload = copy.deepcopy(correlation_agent_common_payload)
    correlation_agent_payload["query"] = f"Where did change happen for drg {drg['name']}? It {drg['insight'].lower()} by {drg['percentage_change']}"
    correlation_agent_payload["context"]["filters"].append({
        "field": "drg_name",
        "operator": "=",
        "value": drg["name"],
        "source": "dimension_match"
    })
    print(f"Calling Correlation Agent for drg '{drg['name']}'...")
    correlation_response = requests.post(correlation_agent_url, json=correlation_agent_payload)
    correlation_result = correlation_response.json()

    print(f"Status: {correlation_response.status_code}")
    print(f"Success: {correlation_result.get('status', False)}")
    correlation_results["drgs"][drg["name"]] = correlation_result

Calling Correlation Agent for state 'ME'...
Status: 200
Success: success
Calling Correlation Agent for state 'CO'...
Status: 200
Success: success
Calling Correlation Agent for provider 'EMORY HILLANDALE HOSPITAL'...
Status: 200
Success: success
Calling Correlation Agent for provider 'EMORY DECATUR HOSPITAL'...
Status: 200
Success: success
Calling Correlation Agent for provider 'YALE NEW HAVEN HOSPITAL'...
Status: 200
Success: success
Calling Correlation Agent for provider 'HENRICO DOCTORS' HOSPITAL'...
Status: 200
Success: success
Calling Correlation Agent for drg 'Ungroupable'...
Status: 200
Success: success
Calling Correlation Agent for drg 'Chemotherapy without Acute Leukemia as Secondary Diagnosis with MCC'...
Status: 200
Success: success
Calling Correlation Agent for drg 'Tendonitis, Myositis and Bursitis without MCC'...
Status: 200
Success: success
Calling Correlation Agent for drg 'Cesarean Section without Sterilization without CC/MCC'...
Status: 200
Success: success
Calling Cor

In [18]:
# store this for use in future cells
CONVERSATION_ID = conversation_id

In [ ]:
# fname = f"anomaly{datetime.now().strftime('%Y%m%d')}.json"
# print(fname)
# with open(fname, 'w') as f:
#     json.dump(anomaly_json, f, indent=4)  # indent makes it human-readable

In [ ]:
fname = f"correlation_results_{datetime.now().strftime('%Y%m%d')}_{conversation_id.replace("tutorial-", "")}.json"
print(fname)
with open(fname, 'w') as f:
    json.dump(correlation_results, f, indent=4)  # indent makes it human-readable
    
# with open("correlation_results_20260514_IP_AUTH-Commercial-202604-R3-202601.json", 'r') as f:
#     correlation_results = json.load(f)

## Step 2.2: Pattern Analysis Agent

The Pattern Analysis Agent extracts meaningful patterns from the DEEP_DIVE report, identifying key trends, outliers, and actionable insights.


In [ ]:
# Call Pattern Analysis Agent
pattern_agent_url = f"{API_BASE_URL}/agents/pattern_agent"
pattern_request = {
    "conversation_id": CONVERSATION_ID,
    "query": "Summarize the highest-impact authorization and provider mix themes from the completed correlation analysis.",
    "context": {
        "anomaly_context" : anomaly_json,
        "deep_dive_report": deep_dive_json,
        "correlation_results": correlation_results
    }
}

print("Calling Pattern Analysis Agent...")
pattern_response = requests.post(pattern_agent_url, json=pattern_request)
pattern_results = pattern_response.json()

print(f"Status: {pattern_response.status_code}")
print(f"Success: {pattern_results.get('status', False)}")

Calling Pattern Analysis Agent...
Status: 200
Success: success


In [82]:
# print(json.dumps(pattern_result, indent=2))

In [ ]:
fname = f"pattern_results_{datetime.now().strftime('%Y%m%d')}_{conversation_id.replace('tutorial-', '')}.json"
print(fname)
with open(fname, 'w') as f:
    json.dump(pattern_results, f, indent=4)  # indent makes it human-readable
    
    
# with open("pattern_results_20260527_IP_AUTH-Commercial-202604-R3-202601.json", 'r') as f:
#     pattern_results = json.load(f)

## Step 3: Reimbursement Agent

The Reimbursement Agent analyzes payer policies to understand reimbursement rules, bundling logic, and denial conditions for identified DRG codes. This provides context for understanding cost drivers and potential interventions.

In [20]:
# Load pattern results
import json
 
with open('pattern_results_20260527_IP_AUTH-Commercial-202604-R3-202601.json', 'r') as f:
    pattern_results = json.load(f)
 
# Extract pattern 5 and cards
patterns = pattern_results['output']['business_patterns']
pattern_5 = next(p for p in patterns if p['pattern_rank'] == 5)
 
# Get all cards (needed for LOB/Product extraction)
all_cards = pattern_results['output'].get('cards', [])
 
# Create the complete input payload
payload = {
    "context": {
        "pattern": pattern_5,
        "cards": all_cards  # ← CRITICAL for LOB/Product extraction!
    },
    "conversation_id": "test-pattern-5-cesarean",
    "query": "Analyze reimbursement policies for pattern 5: California cesarean delivery",
    "job_id": "test-pattern-5-20260528"
}

In [42]:
print(json.dumps(payload, indent=2))

{
  "context": {
    "pattern": {
      "pattern_rank": 5,
      "top_pattern": "California cesarean delivery spend is rising through more complex OB mix",
      "pattern_type": "clinical_case_mix",
      "what_is_impacting": "Commercial maternity / Inpatient OB delivery / Cesarean section",
      "priority_entities": {
        "states": [
          "CA"
        ],
        "providers": [
          "UCHEALTH MEMORIAL HOSPITAL CENTRAL"
        ],
        "products": [],
        "facility_types": [
          "ACUTE HOSPITAL"
        ],
        "clinical_categories": [
          "IP OB Dlvry/Well NB"
        ]
      },
      "key_driver_codes": [
        "Cesarean Section without Sterilization without CC/MCC",
        "PRE-EXISTING HTN PRE-ECLAMP COMP CB",
        "POST-TERM PREGNANCY",
        "MAT CARE LW TRANS SCAR PREV C/S DEL",
        "TWIN PG CHORIONIC/MONOAMNIOT 2ND TM"
      ],
      "impact_summary": {
        "primary_metric": "total_paid",
        "direction": "increase",
     

In [36]:
# Call Reimbursment Agent
reimbursement_agent_url = f"{API_BASE_URL}/agents/reimbursement_policy"
reimbursement_request = payload

print("Calling Reimbursment Agent...")
reimbursement_response = requests.post(reimbursement_agent_url, json=reimbursement_request)
reimbursement_results = reimbursement_response.json()

print(f"Status: {reimbursement_response.status_code}")
print(f"Success: {reimbursement_results.get('status', False)}")

Calling Reimbursment Agent...
Status: 200
Success: success


In [37]:
reimbursement_results

{'job_id': 'test-pattern-5-20260528',
 'conversation_id': 'test-pattern-5-cesarean',
 'agent': 'reimbursement_policy',
 'status': 'success',
 'output': {'pattern_rank': 5,
  'summary_table': {'title': 'Payer Policy Summary',
   'subtitle': 'California cesarean delivery spend is rising through more complex OB mix',
   'columns': [{'id': 'payer_org',
     'label': 'Payer Organization',
     'type': 'text'},
    {'id': 'maternity_admission_auth',
     'label': 'Maternity Admit Auth',
     'type': 'text'},
    {'id': 'multiple_birth_bundling',
     'label': 'Multiple Birth Rules',
     'type': 'text'},
    {'id': 'appeals_process',
     'label': 'Appeals Process\n(Documented)',
     'type': 'badge'},
    {'id': 'policy_effective_date',
     'label': 'Policy Effective Date\n(Last Updated)',
     'type': 'date'}],
   'rows': [{'payer_org': 'Cigna',
     'appeals_process': '-',
     'policy_effective_date': 'N/A',
     'maternity_admission_auth': 'No maternity admit authorization requirement 

In [ ]:
fname = f"reimbursement_results_{datetime.now().strftime('%Y%m%d')}_{conversation_id.replace("tutorial-", "")}.json"
print(fname)
with open(fname, 'w') as f:
    json.dump(reimbursement_results, f, indent=4)  # indent makes it human-readable
    
# with open("correlation_results_20260514_IP_AUTH-Commercial-202604-R3-202601.json", 'r') as f:
#     correlation_results = json.load(f)

## Step 4: Recommendation Agent

The Recommendation Agent synthesizes insights from pattern analysis and reimbursement policies to generate actionable recommendations. It uses decision tree rules and UI-optimized prompts to ensure concise, specific output.

In [5]:
# Recommendation Agent Configuration
from deep_research_agents.decision_tree_rules import DecisionTreeRuleEngine

# Decision tree rules path
DECISION_TREE_PATH = project_root / "configs" / "decision_tree_rules.yaml"

# UI Display Constraints (Word Limits)
MAX_DESCRIPTION_WORDS = 100
MAX_EVIDENCE_WORDS = 15
MAX_STORY_ALIGNMENT_WORDS = 30
MAX_PEER_BENCHMARKING_WORDS = 25

# Validation Settings
REQUIRE_SPECIFICITY = True
SPECIFICITY_THRESHOLD = 0.75
MAX_RETRY_ATTEMPTS = 2

print("Configuration:")
print(f"  Decision Tree: {DECISION_TREE_PATH}")
print(f"  Max Description: {MAX_DESCRIPTION_WORDS} words")
print(f"  Max Evidence: {MAX_EVIDENCE_WORDS} words")
print(f"  Specificity Required: {REQUIRE_SPECIFICITY}")

Configuration:
  Decision Tree: c:\projects\coc\dev\idiscovery-deep-research\configs\decision_tree_rules.yaml
  Max Description: 100 words
  Max Evidence: 15 words
  Specificity Required: True


In [6]:
# Load decision tree rules
if DECISION_TREE_PATH.exists():
    try:
        rule_engine = DecisionTreeRuleEngine(str(DECISION_TREE_PATH))
        rule_count = rule_engine.get_total_rule_count()
        category_count = len(rule_engine.get_category_names())
        print(f"✓ Decision tree rules loaded")
        print(f"  Total rules: {rule_count}")
        print(f"  Categories: {category_count}")
        
        # Format rules for LLM
        rules_text = rule_engine.format_rules_compact()
        print(f"  Formatted for LLM: {len(rules_text)} characters")
    except Exception as e:
        print(f"❌ Failed to load decision tree: {e}")
        rule_engine = None
        rules_text = ""
else:
    print(f"⚠ Decision tree YAML not found at: {DECISION_TREE_PATH}")
    rule_engine = None
    rules_text = ""

2026-05-29 14:12:36,062 - deep_research_agents.decision_tree_rules - INFO - Loaded 950 rules from 18 categories
✓ Decision tree rules loaded
  Total rules: 950
  Categories: 18
  Formatted for LLM: 139598 characters


In [1]:
import json

# Open the file and load its content
with open('pattern_results_20260520_IP_AUTH-Commercial-202604-R3-202601.json', 'r') as file:
    pattern_results = json.load(file)

print(pattern_results)

{'job_id': 'fef01248fe1b475ab9163f19de048831', 'conversation_id': 'tutorial-IP_AUTH-Commercial-202604-R3-202601', 'agent': 'pattern_agent', 'status': 'success', 'output': {'business_patterns': [{'pattern_rank': 1, 'top_pattern': 'Authorization coding and facility mapping shifts are distorting the read', 'pattern_type': 'coding_mapping_validation', 'what_is_impacting': 'Commercial inpatient authorization classification and facility mapping', 'priority_entities': {'states': ['CO', 'ME'], 'providers': ['EMORY HILLANDALE HOSPITAL', 'EMORY DECATUR HOSPITAL', "HENRICO DOCTORS' HOSPITAL"], 'products': ['HMO'], 'facility_types': ['Not Mapped'], 'clinical_categories': []}, 'key_driver_codes': ['AUTH_CODE_DISTRIBUTION', '-3', 'Not Mapped', 'PA Y', 'PA N'], 'impact_summary': {'primary_metric': 'total_paid', 'direction': 'mixed', 'estimated_delta': '≈$19.5M affected', 'volume_signal': 'unknown', 'unit_cost_signal': 'unknown'}, 'pattern_details': 'A large share of the apparent cost movement is tied

In [2]:
import json

# Open the file and load its content
with open('reimbursement_results_20260528_IP_AUTH-Commercial-202604-R3-202601.json', 'r') as file:
    reimbursement_results = json.load(file)

print(reimbursement_results)

[{'pattern_rank': 1, 'pattern_title': 'Authorization coding and facility mapping shifts must be cleaned up first', 'pattern_details': 'The biggest cross-cutting issue is movement between authorization buckets rather than a clean business trend. In multiple markets and clinical populations, PA-required and non-PA spend both rose while legacy uncategorized buckets fell sharply, which strongly suggests coding or mapping reclassification. A separate facility mapping problem is also present, with meaningful spend now landing in a not-mapped facility bucket.', 'evidence_summary': ['Colorado shows PA-required spend up $19.3M and non-PA spend up $9.9M while the legacy -3 bucket fell $28.0M.', 'Maine shows the same pattern: PA-required up $10.6M, non-PA up $3.9M, and legacy -3 down $8.3M.', 'Chemotherapy alone shows about $2.2M of net movement with Y and N up while -2 and -3 buckets fell.', 'Colorado added $1.5M in spend to a Not Mapped facility bucket, including 10 admissions at about $146K pe

In [13]:
import json
from pathlib import Path
from typing import Any, Dict, List, Optional


def load_json_file(file_path: str) -> Any:
    """
    Load and parse a JSON file.
    
    Args:
        file_path: Path to the JSON file
        
    Returns:
        Parsed JSON data
        
    Raises:
        FileNotFoundError: If file doesn't exist
        json.JSONDecodeError: If file is not valid JSON
    """
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")
    
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def combine_pattern_reimbursement_from_json(
    pattern_results_path: str,
    reimbursement_results_path: str,
    output_path: Optional[str] = None
) -> Dict[str, Any]:
    """
    Combine pattern analysis results with reimbursement policy data.
    
    This function:
    1. Loads pattern results (from pattern agent)
    2. Loads reimbursement results (from reimbursement policy agent)
    3. Extracts cards and groups from pattern output
    4. Matches patterns by rank
    5. Fetches full card and group objects using source IDs
    6. Combines data for each pattern with full source objects
    7. Removes priority_entities field (not needed for recommendations)
    8. Returns combined data ready for recommendation agent
    
    Args:
        pattern_results_path: Path to pattern agent results JSON
        reimbursement_results_path: Path to reimbursement policy agent results JSON
        output_path: Optional path to save combined results
        
    Returns:
        Combined data dictionary with enriched patterns, including:
        - All pattern fields (pattern_rank, top_pattern, impact_summary, etc.)
        - source_card_ids: List of source card IDs from pattern analysis
        - source_group_ids: List of source group IDs from pattern analysis
        - source_cards: List of full card objects (fetched using source_card_ids)
        - source_groups: List of full group objects (fetched using source_group_ids)
        - reimbursement: Nested object with policy data and source traceability
        
        Note: priority_entities field is excluded from the output
        
    Example:
        >>> combined = combine_pattern_reimbursement(
        ...     "pattern_results.json",
        ...     "reimbursement_results.json"
        ... )
        >>> # Pass to recommendation agent
        >>> from deep_research_agents import RecommendationAgent
        >>> agent = RecommendationAgent()
        >>> recommendations = agent(input_data=combined)
    """
    # Load both result files
    print(f"Loading pattern results from: {pattern_results_path}")
    pattern_data = load_json_file(pattern_results_path)
    
    print(f"Loading reimbursement results from: {reimbursement_results_path}")
    reimbursement_data = load_json_file(reimbursement_results_path)
    
    # Extract patterns from wrapper if present
    if isinstance(pattern_data, dict) and "output" in pattern_data:
        patterns = pattern_data["output"].get("business_patterns", [])
        cards = pattern_data["output"].get("cards", [])
        groups = pattern_data["output"].get("groups", [])
        metadata = {
            "job_id": pattern_data.get("job_id"),
            "conversation_id": pattern_data.get("conversation_id"),
            "agent": pattern_data.get("agent"),
            "status": pattern_data.get("status")
        }
    else:
        patterns = pattern_data if isinstance(pattern_data, list) else []
        cards = []
        groups = []
        metadata = {}
    
    # Create lookup dictionaries for cards and groups by ID
    cards_by_id = {card.get("card_id"): card for card in cards if "card_id" in card}
    groups_by_id = {group.get("group_id"): group for group in groups if "group_id" in group}
    
    # Reimbursement data is a list
    reimbursement_patterns = reimbursement_data if isinstance(reimbursement_data, list) else []
    
    # Create lookup by pattern_rank
    reimbursement_by_rank = {
        item["pattern_rank"]: item 
        for item in reimbursement_patterns 
        if "pattern_rank" in item
    }
    
    print(f"\nFound {len(patterns)} patterns, {len(cards)} cards, {len(groups)} groups, and {len(reimbursement_by_rank)} reimbursement entries")
    
    # Combine data
    combined_patterns = []
    
    for pattern in patterns:
        pattern_rank = pattern.get("pattern_rank")
        
        # Start with pattern data (includes all fields including source_card_ids and source_group_ids)
        combined = dict(pattern)
        
        # Remove priority_entities field
        combined.pop("priority_entities", None)
        
        # Fetch source cards and groups using IDs
        source_card_ids = pattern.get("source_card_ids", [])
        source_group_ids = pattern.get("source_group_ids", [])
        
        # Fetch full card objects
        source_cards = [cards_by_id[card_id] for card_id in source_card_ids if card_id in cards_by_id]
        
        # Fetch full group objects
        source_groups = [groups_by_id[group_id] for group_id in source_group_ids if group_id in groups_by_id]
        
        # Add source traceability with full objects
        combined["source_card_ids"] = source_card_ids
        combined["source_group_ids"] = source_group_ids
        combined["source_cards"] = source_cards
        combined["source_groups"] = source_groups
        
        # Add reimbursement data if available
        if pattern_rank in reimbursement_by_rank:
            reimbursement = reimbursement_by_rank[pattern_rank]
            
            # Add reimbursement-specific fields
            combined["reimbursement"] = {
                "pattern_title": reimbursement.get("pattern_title"),
                "pattern_details": reimbursement.get("pattern_details"),
                "evidence_summary": reimbursement.get("evidence_summary", []),
                "individual_policies": reimbursement.get("individual_policies", []),
                "summary_table": reimbursement.get("summary_table", {}),
                "recommendation": reimbursement.get("recommendation"),
                # Include source traceability from reimbursement if available
                "source_card_ids": reimbursement.get("source_card_ids", []),
                "source_group_ids": reimbursement.get("source_group_ids", [])
            }
            
            print(f"  ✓ Pattern {pattern_rank}: Combined with reimbursement data ({len(source_cards)} cards, {len(source_groups)} groups)")
        else:
            print(f"  ⚠ Pattern {pattern_rank}: No reimbursement data found ({len(source_cards)} cards, {len(source_groups)} groups)")
            combined["reimbursement"] = None
        
        combined_patterns.append(combined)
    
    # Build final output structure
    result = {
        "metadata": metadata,
        "patterns_data": combined_patterns,
        "summary": {
            "total_patterns": len(combined_patterns),
            "patterns_with_reimbursement": sum(1 for p in combined_patterns if p.get("reimbursement")),
            "patterns_without_reimbursement": sum(1 for p in combined_patterns if not p.get("reimbursement"))
        }
    }
    
    # Save to file if requested
    if output_path:
        output_file = Path(output_path)
        output_file.parent.mkdir(parents=True, exist_ok=True)
        
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(result, f, indent=2, ensure_ascii=False)
        
        print(f"\n✓ Combined results saved to: {output_path}")
    
    print(f"\n✓ Combined {result['summary']['total_patterns']} patterns")
    print(f"  - {result['summary']['patterns_with_reimbursement']} with reimbursement data")
    print(f"  - {result['summary']['patterns_without_reimbursement']} without reimbursement data")
    
    return result


def combine_pattern_reimbursement(
    pattern_data,
    reimbursement_data
) -> Dict[str, Any]:
    """
    Combine pattern analysis results with reimbursement policy data.
    
    This function:
    1. Loads pattern results (from pattern agent)
    2. Loads reimbursement results (from reimbursement policy agent)
    3. Extracts cards and groups from pattern output
    4. Matches patterns by rank
    5. Fetches full card and group objects using source IDs
    6. Combines data for each pattern with full source objects
    7. Removes priority_entities field (not needed for recommendations)
    8. Returns combined data ready for recommendation agent
    
    Args:
        pattern_results_path: Path to pattern agent results JSON
        reimbursement_results_path: Path to reimbursement policy agent results JSON
        output_path: Optional path to save combined results
        
    Returns:
        Combined data dictionary with enriched patterns, including:
        - All pattern fields (pattern_rank, top_pattern, impact_summary, etc.)
        - source_card_ids: List of source card IDs from pattern analysis
        - source_group_ids: List of source group IDs from pattern analysis
        - source_cards: List of full card objects (fetched using source_card_ids)
        - source_groups: List of full group objects (fetched using source_group_ids)
        - reimbursement: Nested object with policy data and source traceability
        
        Note: priority_entities field is excluded from the output
        
    Example:
        >>> combined = combine_pattern_reimbursement(
        ...     "pattern_results.json",
        ...     "reimbursement_results.json"
        ... )
        >>> # Pass to recommendation agent
        >>> from deep_research_agents import RecommendationAgent
        >>> agent = RecommendationAgent()
        >>> recommendations = agent(input_data=combined)
    """
    
    # Extract patterns from wrapper if present
    if isinstance(pattern_data, dict) and "output" in pattern_data:
        patterns = pattern_data["output"].get("business_patterns", [])
        cards = pattern_data["output"].get("cards", [])
        groups = pattern_data["output"].get("groups", [])
        metadata = {
            "job_id": pattern_data.get("job_id"),
            "conversation_id": pattern_data.get("conversation_id"),
            "agent": pattern_data.get("agent"),
            "status": pattern_data.get("status")
        }
    else:
        patterns = pattern_data if isinstance(pattern_data, list) else []
        cards = []
        groups = []
        metadata = {}
    
    # Create lookup dictionaries for cards and groups by ID
    cards_by_id = {card.get("card_id"): card for card in cards if "card_id" in card}
    groups_by_id = {group.get("group_id"): group for group in groups if "group_id" in group}
    
    # Reimbursement data is a list
    reimbursement_patterns = reimbursement_data if isinstance(reimbursement_data, list) else []
    
    # Create lookup by pattern_rank
    reimbursement_by_rank = {
        item["pattern_rank"]: item 
        for item in reimbursement_patterns 
        if "pattern_rank" in item
    }
    
    print(f"\nFound {len(patterns)} patterns, {len(cards)} cards, {len(groups)} groups, and {len(reimbursement_by_rank)} reimbursement entries")
    
    # Combine data
    combined_patterns = []
    
    for pattern in patterns:
        pattern_rank = pattern.get("pattern_rank")
        
        # Start with pattern data (includes all fields including source_card_ids and source_group_ids)
        combined = dict(pattern)
        
        # Remove priority_entities field
        combined.pop("priority_entities", None)
        
        # Fetch source cards and groups using IDs
        source_card_ids = pattern.get("source_card_ids", [])
        source_group_ids = pattern.get("source_group_ids", [])
        
        # Fetch full card objects
        source_cards = [cards_by_id[card_id] for card_id in source_card_ids if card_id in cards_by_id]
        
        # Fetch full group objects
        source_groups = [groups_by_id[group_id] for group_id in source_group_ids if group_id in groups_by_id]
        
        # Add source traceability with full objects
        combined["source_card_ids"] = source_card_ids
        combined["source_group_ids"] = source_group_ids
        combined["source_cards"] = source_cards
        combined["source_groups"] = source_groups
        
        # Add reimbursement data if available
        if pattern_rank in reimbursement_by_rank:
            reimbursement = reimbursement_by_rank[pattern_rank]
            
            # Add reimbursement-specific fields
            combined["reimbursement"] = {
                "pattern_title": reimbursement.get("pattern_title"),
                "pattern_details": reimbursement.get("pattern_details"),
                "evidence_summary": reimbursement.get("evidence_summary", []),
                "individual_policies": reimbursement.get("individual_policies", []),
                "summary_table": reimbursement.get("summary_table", {}),
                "recommendation": reimbursement.get("recommendation"),
                # Include source traceability from reimbursement if available
                "source_card_ids": reimbursement.get("source_card_ids", []),
                "source_group_ids": reimbursement.get("source_group_ids", [])
            }
            
            print(f"  ✓ Pattern {pattern_rank}: Combined with reimbursement data ({len(source_cards)} cards, {len(source_groups)} groups)")
        else:
            print(f"  ⚠ Pattern {pattern_rank}: No reimbursement data found ({len(source_cards)} cards, {len(source_groups)} groups)")
            combined["reimbursement"] = None
        
        combined_patterns.append(combined)
    
    # Build final output structure
    result = {
        "metadata": metadata,
        "patterns_data": combined_patterns,
        "summary": {
            "total_patterns": len(combined_patterns),
            "patterns_with_reimbursement": sum(1 for p in combined_patterns if p.get("reimbursement")),
            "patterns_without_reimbursement": sum(1 for p in combined_patterns if not p.get("reimbursement"))
        }
    }
    
    print(f"\n✓ Combined {result['summary']['total_patterns']} patterns")
    print(f"  - {result['summary']['patterns_with_reimbursement']} with reimbursement data")
    print(f"  - {result['summary']['patterns_without_reimbursement']} without reimbursement data")
    
    return result

In [14]:
# pattern_path = "pattern_results_20260520_IP_AUTH-Commercial-202604-R3-202601.json"
# reimbursement_path = "reimbursement_results_20260528_IP_AUTH-Commercial-202604-R3-202601.json"
# output_path = "combined_pattern_reimbursement_results.json"

# # Option 1: Just combine the data
# payload = combine_pattern_reimbursement_from_json(
#     pattern_path,
#     reimbursement_path,
#     output_path
# )

In [15]:
payload = combine_pattern_reimbursement(
    pattern_results,
    reimbursement_results
)


Found 8 patterns, 85 cards, 24 groups, and 8 reimbursement entries
  ✓ Pattern 1: Combined with reimbursement data (4 cards, 2 groups)
  ✓ Pattern 2: Combined with reimbursement data (4 cards, 2 groups)
  ✓ Pattern 3: Combined with reimbursement data (4 cards, 5 groups)
  ✓ Pattern 4: Combined with reimbursement data (4 cards, 4 groups)
  ✓ Pattern 5: Combined with reimbursement data (4 cards, 4 groups)
  ✓ Pattern 6: Combined with reimbursement data (3 cards, 2 groups)
  ✓ Pattern 7: Combined with reimbursement data (4 cards, 3 groups)
  ✓ Pattern 8: Combined with reimbursement data (2 cards, 2 groups)

✓ Combined 8 patterns
  - 8 with reimbursement data
  - 0 without reimbursement data


In [16]:
print(json.dumps(payload, indent=2))

{
  "metadata": {
    "job_id": "fef01248fe1b475ab9163f19de048831",
    "conversation_id": "tutorial-IP_AUTH-Commercial-202604-R3-202601",
    "agent": "pattern_agent",
    "status": "success"
  },
  "patterns_data": [
    {
      "pattern_rank": 1,
      "top_pattern": "Authorization coding and facility mapping shifts are distorting the read",
      "pattern_type": "coding_mapping_validation",
      "what_is_impacting": "Commercial inpatient authorization classification and facility mapping",
      "key_driver_codes": [
        "AUTH_CODE_DISTRIBUTION",
        "-3",
        "Not Mapped",
        "PA Y",
        "PA N"
      ],
      "impact_summary": {
        "primary_metric": "total_paid",
        "direction": "mixed",
        "estimated_delta": "\u2248$19.5M affected",
        "volume_signal": "unknown",
        "unit_cost_signal": "unknown"
      },
      "pattern_details": "A large share of the apparent cost movement is tied to authorization code redistribution rather than a cle

In [ ]:
# Call Reimbursment Agent
recommendation_agent_url = f"{API_BASE_URL}/agents/recommendation_synthesis"
recommendation_request = payload

print("Calling Reimbursment Agent...")
recommendation_response = requests.post(recommendation_agent_url, json=recommendation_request)
recommendation_results = recommendation_response.json()

print(f"Status: {recommendation_response.status_code}")
print(f"Success: {recommendation_results.get('status', False)}")

Calling Reimbursment Agent...
Status: 200
Success: False


In [20]:
recommendation_results

{'success': True,
 'result': {'metadata': {'total_patterns': 8,
   'recommendations_generated': 7,
   'patterns_skipped': 1,
   'approach': 'llm_based_dtr_relevance',
   'dtr_rules_path': 'C:\\projects\\coc\\dev\\idiscovery-deep-research\\configs\\decision_tree_rules.yaml'},
  'recommendations': [{'rank': 2,
    'priority': 'HIGH',
    'category': 'Policy',
    'description': 'Initiate a Commercial HMO acute inpatient contract-remediation and site-of-care policy review for Colorado and Maine, with focused contract compliance audit of PA Y and PA N pathways and provider-level follow-up on Emory Decatur Hospital.',
    'evidence': ['Colorado HMO acute PA Y added $17.6M with 517 more admissions.',
     'Maine HMO acute PA Y added $10.1M with 161 more admissions.',
     'Colorado HMO acute PA N added $8.1M on 397 more admissions.',
     'Emory Decatur added about $1.9M across HMO PA Y and PA N.'],
    'story_alignment': ['Business Rule IP MedSurg (DNE) - Is variance isolated to subset of p

## Summary

This notebook demonstrates the complete end-to-end workflow for deep research analysis:

1. **✓ Data Loading**: Loaded insights from Rajib's anomaly detection (KEY_INSIGHT and DEEP_DIVE)
2. **✓ Correlation Analysis**: Identified additional dimensions (procedures, modifiers) driving patterns
3. **✓ Reimbursement Analysis**: Analyzed payer policies for DRG codes
4. **✓ Recommendation Synthesis**: Generated actionable recommendations combining pattern insights and reimbursement findings

### Key Outputs

- **Correlation Results**: Stored in correlation workflow cells
- **Reimbursement Results**: `reimbursement_agent_payor_summary_*.json`
- **Recommendations**: `recommendations_*.json`

### Data Sources Used for Recommendations

- ✓ Pattern Analysis (KEY_INSIGHT + DEEP_DIVE)
- ✓ Reimbursement Policy Analysis
- ✗ Correlation Analysis (analyzed separately, not fed to recommendations)

### Next Steps

1. Review recommendations for actionability
2. Validate findings with domain experts
3. Prioritize implementation based on estimated impact
4. Monitor outcomes after implementation